## 🤖 Player Profile Clustering AI ⚽

Modelo não supervisionado de clusterização utilizando K-Means para agrupar jogadores de futebol em perfis de desempenho físico com base em métricas de sessão, apoiando decisões táticas e ajustes individualizados de treino.

In [24]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
import onnx
from sklearn.pipeline import Pipeline

# CONFIGURAÇÕES

dataPath = "../data/players.xlsx"
modelPath = "../model/player-profile-clustering.onnx"

metrics = [
    'Distance (m)',
    'High Intensity Running (m)',
    'Sprint Distance (m)',
    'Top Speed (kph)',
    'Accelerations',
    'Decelerations',
    'Metres per Minute (m)'
]

In [25]:
# EXPORTAÇÃO DOS DADOS

df = pd.read_excel(filePath)

ws = df[df['Segment Name'] == 'Whole Session'].copy()
ws = ws[ws['Distance (m)'] > 2000]

match_df = ws.dropna(subset=metrics).copy()

if match_df.shape[0] < 4:
    raise ValueError("Poucos dados para clustering")

In [26]:
# NORMALIZAÇÃO

scaler = StandardScaler()
X = scaler.fit_transform(match_df[metrics])

In [27]:
# TREINAMENTO DO MODELO

km = KMeans(n_clusters=4, random_state=42, n_init=10)
match_df['cluster'] = km.fit_predict(X)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('kmeans', KMeans(n_clusters=4, random_state=42, n_init=10))
])

match_df['cluster'] = pipeline.fit_predict(match_df[metrics])

In [28]:
# SALVAR MODELO

initial_type = [('float_input', FloatTensorType([None, len(metrics)]))]

onnx_model = convert_sklearn(pipeline, initial_types=initial_type)

with open(modelPath, "wb") as f:
    f.write(onnx_model.SerializeToString())